# **Data Preprocessing**

In [23]:
# conda activate "F:\_GitHub\CSCI161-SocialComputing\Binwag - PhilippineRedditAgricultureAnalysis\env"
# OR go into the directory above, and conda activate
# conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia

# conda activate ./env
# jupyter notebook


In [24]:
import os
import sys
import subprocess

print("Checking GPU and environment...")

conda_env = os.environ.get("CONDA_PREFIX", None)
print(f"Conda environment: {conda_env if conda_env else 'Not in Conda'}")

try:
    result = subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if result.returncode == 0:
        print("NVIDIA GPU detected.")
        print(result.stdout.split("\n")[2]) 
    else:
        print("No NVIDIA GPU found or drivers missing.")
except FileNotFoundError:
    print("'nvidia-smi' not found — NVIDIA drivers may not be installed.")

try:
    import torch
    print("PyTorch already installed.")
except ImportError:
    print("Installing PyTorch with CUDA 12.1 support...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "torch", "torchvision", "torchaudio",
        "--extra-index-url", "https://download.pytorch.org/whl/cu121"
    ])
    import torch

try:
    import torch
    print("PyTorch CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Using GPU:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available — running on CPU.")
except Exception as e:
    print("PyTorch check failed:", e)
    print("Trying TensorFlow GPU check instead...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow[and-cuda]"])
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    print("TensorFlow GPUs detected:" if gpus else "TensorFlow found no GPU.", gpus)

print("Setup complete.")


Checking GPU and environment...
Conda environment: F:\_GitHub\CSCI161-SocialComputing\Binwag - PhilippineRedditAgricultureAnalysis\env
NVIDIA GPU detected.
| NVIDIA-SMI 572.16                 Driver Version: 572.16         CUDA Version: 12.8     |
PyTorch already installed.
PyTorch CUDA available: True
Using GPU: NVIDIA GeForce RTX 3070
Setup complete.


In [25]:
import pandas as pd
import os

print(f"Current directory: {os.getcwd()}")

reddit_raw = pd.read_csv(r"data/Reddit/reddit_data.csv")
rappler_raw = pd.read_csv(r"data/News/rappler_agriculture.csv")

reddit_raw.info()
reddit_raw.tail()



Current directory: F:\_GitHub\CSCI161-SocialComputing\Binwag - PhilippineRedditAgricultureAnalysis
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13171 entries, 0 to 13170
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   post_title      13171 non-null  object 
 1   post_url        13171 non-null  object 
 2   comment         12944 non-null  object 
 3   comment_author  12944 non-null  object 
 4   comment_score   12944 non-null  float64
 5   post_body       227 non-null    object 
 6   post_score      227 non-null    float64
 7   post_author     227 non-null    object 
 8   created_utc     13171 non-null  object 
dtypes: float64(2), object(7)
memory usage: 926.2+ KB


,post_title,post_url,comment,comment_author,comment_score,post_body,post_score,post_author,created_utc
13166,"Philippines, Japan deepen ties with $1.5 billi...",https://reddit.com/r/Philippines/comments/1cio...,That’s why fuck the Dutertes,nikolodeon,10.0,NaN,NaN,NaN,2024-05-03 11:30:11
13167,"Philippines, Japan deepen ties with $1.5 billi...",https://reddit.com/r/Philippines/comments/1cio...,Fr? That's scary.,[deleted],2.0,NaN,NaN,NaN,2024-05-03 12:32:41
13168,"Philippines, Japan deepen ties with $1.5 billi...",https://reddit.com/r/Philippines/comments/1cio...,This partnership is very daijoubu,IkigaiSagasu,5.0,NaN,NaN,NaN,2024-05-03 10:33:52
13169,"Philippines, Japan deepen ties with $1.5 billi...",https://reddit.com/r/Philippines/comments/1cio...,"Yeah, there’s a huge connection between corrup...",FilipinxFurry,1.0,NaN,NaN,NaN,2024-05-03 12:36:12
13170,"Philippines, Japan deepen ties with $1.5 billi...",https://reddit.com/r/Philippines/comments/1cio...,"Yup ganyan din ginagawa ng Russia, kaya mas pi...",SuperBombaBoy,1.0,NaN,NaN,NaN,2024-05-03 16:57:35


In [26]:
rappler_raw.info()
rappler_raw.tail()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           121 non-null    object 
 1   link            121 non-null    object 
 2   date_published  121 non-null    object 
 3   author          0 non-null      float64
 4   text            121 non-null    object 
dtypes: float64(1), object(4)
memory usage: 4.9+ KB


,title,link,date_published,author,text
116,ASF spreads to 4 Aklan towns,https://www.rappler.com/philippines/visayas/as...,2023-05-18T09:10:00+08:00,NaN,"AKLAN, Philippines – A month after the first c..."
117,Pangasinan farmers urged to plant early as San...,https://www.rappler.com/philippines/luzon/pang...,2023-05-06T11:32:23+08:00,NaN,"DAGUPAN CITY, Philippines – With El Niño just ..."
118,[ANALYSIS] Should LBP and DBP be merged into a...,https://www.rappler.com/voices/thought-leaders...,2023-03-31T13:24:41+08:00,NaN,"On March 23, Finance Secretary Benjamin Diokno..."
119,Sugar regulation chief Alba resigns,https://www.rappler.com/business/sugar-regulat...,2023-03-24T11:19:13+08:00,NaN,"MANILA, Philippines – Sugar Regulatory Adminis..."
120,North Korea’s Kim orders ‘fundamental transfor...,https://www.rappler.com/world/asia-pacific/nor...,2023-02-28T11:37:49+08:00,NaN,"SEOUL, South Korea – North Korean leaderKim Jo..."


---
**Cleaning Reddit Data**

In [27]:
import pandas as pd
import re

reddit_clean = reddit_raw.drop(columns=['comment_author', 'post_author','post_score','comment_score'])

reddit_clean = reddit_clean.dropna(subset=['post_title', 'comment'], how='all')

# Combine title + body into one field (if body exists)
reddit_clean['post_full'] = reddit_clean.apply(
    lambda row: f"{row['post_title']} {row['post_body']}" if pd.notnull(row['post_body']) else row['post_title'],
    axis=1
)

# Text normalization
def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"http\S+|www\S+", "", text)       # remove URLs
    text = re.sub(r"<.*?>", "", text)                # remove HTML tags
    text = re.sub(r"[\r\n]+", " ", text)             # newlines → space
    text = re.sub(r"\s+", " ", text).strip()         # normalize whitespace
    text = text.lower()                              # lowercase
    text = text.replace("[deleted]", "").replace("[removed]", "")
    return text.strip()

reddit_clean['post_full'] = reddit_clean['post_full'].apply(clean_text)
reddit_clean['comment'] = reddit_clean['comment'].apply(clean_text)

reddit_clean = reddit_clean[(reddit_clean['post_full'].str.len() > 0) | (reddit_clean['comment'].str.len() > 0)]

reddit_clean = reddit_clean.drop_duplicates(subset=['post_url', 'comment'])

reddit_clean = reddit_clean.reset_index(drop=True)

reddit_clean.to_csv("data/Reddit/reddit_clean.csv", index=False, encoding='utf-8-sig')

print(f"Cleaned {len(reddit_clean)} rows saved to reddit_clean.csv")
reddit_clean.tail(3)


Cleaned 12730 rows saved to reddit_clean.csv


,post_title,post_url,comment,post_body,created_utc,post_full
12727,"Philippines, Japan deepen ties with $1.5 billi...",https://reddit.com/r/Philippines/comments/1cio...,this partnership is very daijoubu,NaN,2024-05-03 10:33:52,"philippines, japan deepen ties with $1.5 billi..."
12728,"Philippines, Japan deepen ties with $1.5 billi...",https://reddit.com/r/Philippines/comments/1cio...,"yeah, there’s a huge connection between corrup...",NaN,2024-05-03 12:36:12,"philippines, japan deepen ties with $1.5 billi..."
12729,"Philippines, Japan deepen ties with $1.5 billi...",https://reddit.com/r/Philippines/comments/1cio...,"yup ganyan din ginagawa ng russia, kaya mas pi...",NaN,2024-05-03 16:57:35,"philippines, japan deepen ties with $1.5 billi..."


Reddit is cleaned by normalizing text, duplicates, and removing unnused columns for this study.

In [28]:
rappler_clean = rappler_raw.copy()

# Drop the 'author' column
rappler_clean = rappler_clean.drop(columns=['author'])

# Text normalization function
def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"http\S+|www\S+", "", text)       # remove URLs
    text = re.sub(r"<.*?>", "", text)                # remove HTML tags
    text = re.sub(r"[\r\n]+", " ", text)            # newlines → space
    text = re.sub(r"\s+", " ", text).strip()        # normalize whitespace
    text = text.lower()                              # lowercase
    text = text.replace("[deleted]", "").replace("[removed]", "")
    return text.strip()

rappler_clean['cleaned_text'] = rappler_clean['text'].apply(clean_text)
rappler_clean.to_csv(r"data/News/rappler_clean.csv", index=False)

rappler_clean.tail()

,title,link,date_published,text,cleaned_text
116,ASF spreads to 4 Aklan towns,https://www.rappler.com/philippines/visayas/as...,2023-05-18T09:10:00+08:00,"AKLAN, Philippines – A month after the first c...","aklan, philippines – a month after the first c..."
117,Pangasinan farmers urged to plant early as San...,https://www.rappler.com/philippines/luzon/pang...,2023-05-06T11:32:23+08:00,"DAGUPAN CITY, Philippines – With El Niño just ...","dagupan city, philippines – with el niño just ..."
118,[ANALYSIS] Should LBP and DBP be merged into a...,https://www.rappler.com/voices/thought-leaders...,2023-03-31T13:24:41+08:00,"On March 23, Finance Secretary Benjamin Diokno...","on march 23, finance secretary benjamin diokno..."
119,Sugar regulation chief Alba resigns,https://www.rappler.com/business/sugar-regulat...,2023-03-24T11:19:13+08:00,"MANILA, Philippines – Sugar Regulatory Adminis...","manila, philippines – sugar regulatory adminis..."
120,North Korea’s Kim orders ‘fundamental transfor...,https://www.rappler.com/world/asia-pacific/nor...,2023-02-28T11:37:49+08:00,"SEOUL, South Korea – North Korean leaderKim Jo...","seoul, south korea – north korean leaderkim jo..."


---

Final overview

In [29]:
rappler_clean.info()
rappler_clean.isna().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   title           121 non-null    object
 1   link            121 non-null    object
 2   date_published  121 non-null    object
 3   text            121 non-null    object
 4   cleaned_text    121 non-null    object
dtypes: object(5)
memory usage: 4.9+ KB


title             0
link              0
date_published    0
text              0
cleaned_text      0
dtype: int64

In [30]:
rappler_clean[['text', 'cleaned_text']].head()


,text,cleaned_text
0,"COTABATO CITY, Philippines —Maguindanao del Su...","cotabato city, philippines —maguindanao del su..."
1,"MANILA, Philippines – Chinese smugglers, Phili...","manila, philippines – chinese smugglers, phili..."
2,"MANILA, Philippines – Bounty Fresh is focusing...","manila, philippines – bounty fresh is focusing..."
3,"MANILA, Philippines – The Po family’s agribusi...","manila, philippines – the po family’s agribusi..."
4,The two-tiered specific tax on Sweetened Bever...,the two-tiered specific tax on sweetened bever...
